In [31]:
#all graphs
import json
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import numpy as np

with open("data/Drained_dense_history.json", "r") as file:
    raw_data = json.load(file)

#outputs shortened list with stepnum items
def data_choose(data, stepnum):  
    filtered = []
    for i in np.linspace(0, len(data)-1, stepnum):
        filtered.append(data[int(np.round(i))])
    return filtered

#outputs shortened list with stepnum items with log scale
def data_choose_log(data, stepnum):  
    filtered = []
    last_i = []
    base = np.power(len(data), (1/(stepnum-1)))
    for i in range(stepnum):
        next_i = int(np.round(np.power(base, i)))-1
        while next_i in last_i and not(next_i >= len(data)-2):
            next_i += 1
        filtered.append(data[next_i])
        last_i.append(next_i)
    return filtered
    
# calculates p and q coordinates for yield curve
def yield_curve(pi, M, n=50):
    p = np.linspace(0, pi * np.e, n)
    q = np.zeros_like(p)
    q[1:] = M * p[1:] * (1 + np.log(pi / p[1:]))
    return p, q

data = raw_data
short = 50
# for col in raw_data:            # shortens data set
#     data.update({col: data_choose_log(raw_data[col], short)})
M = 1.45                        # CSL slope
slide_steps = 100                # number of slider steps
initial_step = 0                # step that slider starts on
static_num = 5                  # number of static traces
step_to_indices = {}            # stores and indexes dynamic traces
p_max = 0                       # maximum x axis p value
q_max = max(data["p_kPa"]) * M  # maximum y axis q value
a, b, c = 1.01, 0.087, 0.38     # parameters for curved CSL in void ratio space
lam, gam = 0.07, 0.9           # parameters for semi-log CSL in void ratio space
csl_type = "curved"
test_type = "drained"
pref = 100                      # reference pressure for CSL                       
pressure = []                   # pore water pressure for isotropic conditions
for step in range(len(raw_data["step"])):
    pressure.append(raw_data["p_kPa"][0] + (raw_data["q_kPa"][step]) / 3 - raw_data["p_kPa"][step])
main_x, main_y = 1, 1           # layout positions of graphs (row,col)
a_x, a_y = 1, 2
b_x, b_y = 2, 2
c_x, c_y = 3, 2
# Initialize figure with subplots
fig = make_subplots(
    rows=3, cols=2,
    column_widths=[0.7, 0.3],
    horizontal_spacing=0.08,   # space between columns
    vertical_spacing=0.1,     # space between rows
    specs=[[{"rowspan": 3}, {}],
           [     None,      {}],
           [     None,      {}]],
)

# Static traces - always visible
fig.add_trace(go.Scatter(x=[0, 2.5 * q_max / M], y=[0, 2.5 * q_max], visible=True, name="CSL", mode = "lines", line = dict(dash='dot', color="blue")), row=main_x, col=main_y)
fig.add_trace(go.Scatter(x=data["axial_strain_percent"], y=data["q_kPa"], visible=True, name="Loading Path", showlegend=False, line=dict(color="#282828")), row=a_x, col=a_y)
if(test_type == "drained"):
    fig.add_trace(go.Scatter(x=data["axial_strain_percent"], y=data["volumetric_strain_percent"], visible=True, name="Loading Path", showlegend=False, line=dict(color="#282828")), row=b_x, col=b_y)
elif(test_type == "undrained"):
    fig.add_trace(go.Scatter(x=data["axial_strain_percent"], y=data_choose(pressure, short), visible=True, name="Loading Path", showlegend=False, line=dict(color="#282828")), row=b_x, col=b_y)
ec = []
if(csl_type == "curved"):
    for p in data["p_kPa"]:
        ec.append(a - b * pow((p / pref), c))
elif(csl_type == "semi_log"):
    for p in data["p_kPa"]:
        ec.append(gam - lam * np.log10(p / pref))
fig.add_trace(go.Scatter(x=data["p_kPa"], y=ec, visible=True, name="CSL", showlegend=False, mode = "lines", line = dict(color = "blue", dash='dot')), row=c_x, col=c_y)
fig.add_trace(go.Scatter(x=data["p_kPa"], y=data["void_ratio"], visible=True, name="Loading Path", showlegend=False, line=dict(color="#282828")), row=c_x, col=c_y)

# Slider-controlled traces
for k, step in enumerate(data_choose_log(raw_data["step"], slide_steps)):
    p_yield, q_yield = yield_curve(raw_data["pi_kPa"][step-1], raw_data["Mi"][step-1])
    if max(p_yield) > p_max:
        p_max = max(p_yield)
    idx_start = len(fig.data)
    fig.add_trace(go.Scatter(
        visible = False,
        name = "Yield Surface",
        line=dict(color="purple"),
        x = p_yield,
        y = q_yield
    ),
        row=main_x, col=main_y
    )
    
    current_p = raw_data["p_kPa"][step-1]
    current_q = raw_data["q_kPa"][step-1]
    fig.add_trace(go.Scatter(
        visible = False,
        name = "Current State",
        mode="markers",
        marker=dict(size=10, color="red"),
        x = [current_p],
        y = [current_q]
    ),
    row=main_x, col=main_y      
    )
    
    fig.add_trace(go.Scatter(
        visible = False,
        name = "Loading Path",
        mode = 'lines',
        line=dict(color="#282828"),
        x = raw_data["p_kPa"][0:step-1],
        y = raw_data["q_kPa"][0:step-1]
    ),
    row=main_x, col=main_y      
    )
    
    fig.add_trace(go.Scatter(
        visible = False,
        name = "Image Point",
        mode="markers",
        marker=dict(size=10, color="green"),
        x = [raw_data["pi_kPa"][step-1]],
        y = [raw_data["pi_kPa"][step-1] * raw_data["Mi"][step-1]]
    ),
    row=main_x, col=main_y             
    )
    
    fig.add_trace(go.Scatter(
        visible = False,
        name = "Current State",
        mode="markers",
        marker=dict(
            size=12,
            symbol="circle-open",
            color="red",       
            line=dict(width=2)
        ),
        showlegend=False,
        x = [raw_data["axial_strain_percent"][step-1]],
        y = [raw_data["q_kPa"][step-1]]
    ),
    row=a_x, col=a_y            
    )
    if(test_type == "drained"):
        fig.add_trace(go.Scatter(
            visible = False,
            name = "Current State",
            mode="markers",
            marker=dict(
                size=12,
                symbol="circle-open",
                color="red",       
                line=dict(width=2)
            ),
            showlegend=False,
            x = [raw_data["axial_strain_percent"][step-1]],
            y = [raw_data["volumetric_strain_percent"][step-1]]
        ),
        row=b_x, col=b_y             
        )
    elif(test_type == "undrained"):    
        fig.add_trace(go.Scatter(
            visible = False,
            name = "Current State",
            mode="markers",
            marker=dict(
                size=12,
                symbol="circle-open",
                color="red",       
                line=dict(width=2)
            ),
            showlegend=False,
            x = [raw_data["axial_strain_percent"][step-1]],
            y = [pressure[step-1]]
        ),
        row=b_x, col=b_y            
        )
    fig.add_trace(go.Scatter(
        visible = False,
        name = "Current State",
        mode="markers",
        marker=dict(
            size=12,
            symbol="circle-open",
            color="red",       
            line=dict(width=2)
        ),
        showlegend=False,
        x = [raw_data["p_kPa"][step-1]],
        y = [raw_data["void_ratio"][step-1]]
    ),
    row=c_x, col=c_y             
    )
    idx_end = len(fig.data)
    step_to_indices[k] = list(range(idx_start, idx_end))

# Initial visible traces
for i in step_to_indices[initial_step]: 
    fig.data[i].visible = True 

# updates visible traces based on slider
steps = []
for k in range(len(step_to_indices)):
    visible = [True] * static_num + [False] * (len(fig.data) - static_num)
    for i in step_to_indices[k]:
        visible[i] = True
    steps.append(dict(method="update", args=[{"visible": visible}], label=f"{fig.data[step_to_indices[k][5]].x[0]:.3f}%"))
    
sliders = [dict(
    active=initial_step, 
    steps=steps,       
    currentvalue={"prefix": "Axial Strain: "}
    )]
fig.update_layout(
    uirevision="constant",
    sliders=sliders
)

#formatting and calculating axis ranges
q_range = [min(data["q_kPa"]), max(data["q_kPa"])]
q_buffer = 0.15 * (max(data["q_kPa"]) - min(data["q_kPa"]))
ec_range = [min(min(ec), min(data["void_ratio"])), max(max(ec), max(data["void_ratio"]))]
ec_buffer = 0.15 * (max(max(ec), max(data["void_ratio"])) - min(min(ec), min(data["void_ratio"])))
v_range = [min(data["volumetric_strain_percent"]), max(data["volumetric_strain_percent"])]
v_buffer = 0.15 * (max(data["volumetric_strain_percent"]) - min(data["volumetric_strain_percent"]))
a_range = [min(data["axial_strain_percent"]), max(data["axial_strain_percent"])]
lnp_range = np.log10([min(data["p_kPa"]), max(data["p_kPa"])])
pr_range = [min(pressure), max(pressure)]
pr_buffer = 0.15 * (max(pressure) - min(pressure))
#a_buffer = 0.15 * (max(data["axial_strain_percent"]) - min(data["axial_strain_percent"]))

fig.update_yaxes(title_text="Deviatoric stress, q (kPa)", range=[0,  2.5 * q_max], row=main_x, col=main_y)
fig.update_xaxes(title_text="Mean effective stress, p' (kPa)", range=[0,  1.15 * p_max], row=main_x, col=main_y)

fig.update_yaxes(title_text="q (kPa)", range=[q_range[0], q_range[1] + q_buffer], row=a_x, col=a_y)
fig.update_xaxes(title_text="Axial strain, ε<sub>1</sub> (%)", title_standoff=0, range=a_range, row=a_x, col=a_y)

if(test_type == "drained"):
    fig.update_yaxes(title_text="Volumetric strain, ε<sub>V</sub> (%)", range=[v_range[0], v_range[1] + v_buffer], row=b_x, col=b_y)
    fig.update_xaxes(title_text="Axial strain, ε<sub>1</sub> (%)", title_standoff=0, range=a_range, row=b_x, col=b_y)
elif(test_type == "undrained"):
    fig.update_yaxes(title_text="Pore-pressure, Δu (kPa)", range=[pr_range[0], pr_range[1] + pr_buffer], row=b_x, col=b_y)
    fig.update_xaxes(title_text="Axial strain, ε<sub>1</sub> (%)", title_standoff=0, range=a_range, row=b_x, col=b_y)
    
fig.update_yaxes(title_text="Void ratio, e", range=[ec_range[0] - ec_buffer, ec_range[1] + ec_buffer], row=c_x, col=c_y)  
fig.update_xaxes(title_text="Mean effective stress, p'(kPa, log scale)", type="log", range=lnp_range, title_standoff=0, row=c_x, col=c_y)

fig.update_layout(
    width=900,
    height=600,
    template = "plotly_white",
    font=dict(size=10),
    margin=dict(l=30, r=20, t=30, b=30),
    legend=dict(
        font=dict(size=9),
        x=1.02, y=1, 
        xanchor="left", yanchor="top",
        bgcolor="rgba(255,255,255,0.8)", 
        bordercolor="#ddd", 
        borderwidth=1)
)
fig.update_xaxes(
    tickfont=dict(size=9),
)
fig.update_yaxes(
    tickfont=dict(size=9),
)
fig.update_traces(marker=dict(size=7), line=dict(width=2))
fig.show(renderer="browser")
#fig.write_html("Drained_Dense_Log.html")

In [25]:
#Drained loose graphs
import json
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import numpy as np

with open("data/Drained_loose_history.json", "r") as file:
    raw_data = json.load(file)

# outputs shortened list with stepnum items
# def data_choose(data, stepnum):  
#     filtered = []
#     for i in np.linspace(0, len(data)-1, stepnum):
#         filtered.append(data[int(np.round(i))])
#     return filtered

#outputs shortened list with stepnum items with log scale
def data_choose(data, stepnum):  
    filtered = []
    base = np.power(len(data), (1/(stepnum-1)))
    for i in range(stepnum):
        filtered.append(data[int(np.round(np.power(base, i)))-1])
    return filtered

# calculates p and q coordinates for yield curve
def yield_curve(pi, M, n=200):
    p = np.linspace(0, pi * np.e, n)
    q = np.zeros_like(p)
    q[1:] = M * p[1:] * (1 + np.log(pi / p[1:]))
    return p, q

data= {}
for col in raw_data:            # slices data set
    data.update({col: raw_data[col][:]})          
M = 1.45                        # CSL slope
slide_steps = 18                # number of slider steps
initial_step = 0                # step that slider starts on
static_num = 5                  # number of static traces
step_to_indices = {}            # stores and indexes dynamic traces
p_path = []                     # loading path q values
q_path = []                     # loading path q values
p_max = 0                       # maximum x axis p value
q_max = max(data["p_kPa"]) * M  # maximum y axis q value
a, b, c = 1.01, 0.087, 0.38     # parameters for curved CSL in void ratio space
pref = 100                      # reference pressure for CSL                       
pressure = []                   # pore water pressure for isotropic conditions
for step in data["step"]:
    pressure.append(data["p_kPa"][0] + (data["q_kPa"][step-1]) / 3 - data["p_kPa"][step-1])
main_x, main_y = 1, 1           # layout positions of graphs (row,col)
a_x, a_y = 1, 2
b_x, b_y = 2, 2
c_x, c_y = 3, 2

# Initialize figure with subplots
fig = make_subplots(
    rows=3, cols=2,
    column_widths=[0.7, 0.3],
    horizontal_spacing=0.08,   # space between columns
    vertical_spacing=0.1,     # space between rows
    specs=[[{"rowspan": 3}, {}],
           [     None,      {}],
           [     None,      {}]],
)

# Static traces - always visible
fig.add_trace(go.Scatter(x=[0, 2.5 * q_max / M], y=[0, 2.5 * q_max], visible=True, name="CSL", mode = "lines", line = dict(dash='dot', color="blue")), row=main_x, col=main_y)
fig.add_trace(go.Scatter(x=data["axial_strain_percent"], y=data["q_kPa"], visible=True, name="Loading Path", showlegend=False, line=dict(color="#282828")), row=a_x, col=a_y)
fig.add_trace(go.Scatter(x=data["axial_strain_percent"], y=data["volumetric_strain_percent"], visible=True, name="Loading Path", showlegend=False, line=dict(color="#282828")), row=b_x, col=b_y)
ec = []
for p in data["p_kPa"]:
    ec.append(a - b * pow((p / pref), c))
fig.add_trace(go.Scatter(x=data["p_kPa"], y=ec, visible=True, name="CSL", showlegend=False, mode = "lines", line = dict(color = "blue", dash='dot')), row=c_x, col=c_y)
fig.add_trace(go.Scatter(x=data["p_kPa"], y=data["void_ratio"], visible=True, name="Loading Path", showlegend=False, line=dict(color="#282828")), row=c_x, col=c_y)

# Slider-controlled traces
for k, step in enumerate(data_choose(data["step"], slide_steps)):
    p_yield, q_yield = yield_curve(data["pi_kPa"][step-1], M)
    if max(p_yield) > p_max:
        p_max = max(p_yield)
    idx_start = len(fig.data)
    fig.add_trace(go.Scatter(
        visible = False,
        name = "Yield Surface",
        line=dict(color="purple"),
        x = p_yield,
        y = q_yield
    ),
        row=main_x, col=main_y
    )
    
    current_p = data["p_kPa"][step-1]
    current_q = data["q_kPa"][step-1]
    for i in range(step):
        p_path = data["p_kPa"][0:i+1]
        q_path = data["q_kPa"][0:i+1]
    fig.add_trace(go.Scatter(
        visible = False,
        name = "Current State",
        mode="markers",
        marker=dict(size=10, color="red"),
        x = [current_p],
        y = [current_q]
    ),
    row=main_x, col=main_y      
    )
    
    fig.add_trace(go.Scatter(
        visible = False,
        name = "Loading Path",
        mode = 'lines',
        line=dict(color="#282828"),
        x = p_path,
        y = q_path
    ),
    row=main_x, col=main_y      
    )
    
    fig.add_trace(go.Scatter(
        visible = False,
        name = "Image Point",
        mode="markers",
        marker=dict(size=10, color="green"),
        x = [data["pi_kPa"][step-1]],
        y = [data["pi_kPa"][step-1] * M]
    ),
    row=main_x, col=main_y             
    )
    
    fig.add_trace(go.Scatter(
        visible = False,
        name = "Current State",
        mode="markers",
        marker=dict(
            size=12,
            symbol="circle-open",
            color="red",       
            line=dict(width=2)
        ),
        showlegend=False,
        x = [data["axial_strain_percent"][step-1]],
        y = [data["q_kPa"][step-1]]
    ),
    row=a_x, col=a_y            
    )
    
    fig.add_trace(go.Scatter(
        visible = False,
        name = "Current State",
        mode="markers",
        marker=dict(
            size=12,
            symbol="circle-open",
            color="red",       
            line=dict(width=2)
        ),
        showlegend=False,
        x = [data["axial_strain_percent"][step-1]],
        y = [data["volumetric_strain_percent"][step-1]]
    ),
    row=b_x, col=b_y             
    )
    
    fig.add_trace(go.Scatter(
        visible = False,
        name = "Current State",
        mode="markers",
        marker=dict(
            size=12,
            symbol="circle-open",
            color="red",       
            line=dict(width=2)
        ),
        showlegend=False,
        x = [data["p_kPa"][step-1]],
        y = [data["void_ratio"][step-1]]
    ),
    row=c_x, col=c_y             
    )
    idx_end = len(fig.data)
    step_to_indices[k] = list(range(idx_start, idx_end))

# Initial visible traces
for i in step_to_indices[initial_step]: 
    fig.data[i].visible = True 

# updates visible traces based on slider
steps = []
for k in range(len(step_to_indices)):
    visible = [True] * static_num + [False] * (len(fig.data) - static_num)
    for i in step_to_indices[k]:
        visible[i] = True
    steps.append(dict(method="update", args=[{"visible": visible}], label=f"{fig.data[step_to_indices[k][5]].x[0]:.3f}%"))
    
sliders = [dict(
    active=initial_step, 
    steps=steps, 
    currentvalue={"prefix": "Axial Strain: "}
    )]
fig.update_layout(
    uirevision="constant",
    sliders=sliders
)

#formatting and calculating axis ranges
q_range = [min(data["q_kPa"]), max(data["q_kPa"])]
q_buffer = 0.15 * (max(data["q_kPa"]) - min(data["q_kPa"]))
ec_range = [min(min(ec), min(data["void_ratio"])), max(max(ec), max(data["void_ratio"]))]
ec_buffer = 0.15 * (max(max(ec), max(data["void_ratio"])) - min(min(ec), min(data["void_ratio"])))
v_range = [min(data["volumetric_strain_percent"]), max(data["volumetric_strain_percent"])]
v_buffer = 0.15 * (max(data["volumetric_strain_percent"]) - min(data["volumetric_strain_percent"]))
a_range = [min(data["axial_strain_percent"]), max(data["axial_strain_percent"])]
lnp_range = np.log10([min(data["p_kPa"]), max(data["p_kPa"])])
#a_buffer = 0.15 * (max(data["axial_strain_percent"]) - min(data["axial_strain_percent"]))

fig.update_yaxes(title_text="Deviatoric stress, q (kPa)", range=[0,  2.5 * q_max], row=main_x, col=main_y)
fig.update_xaxes(title_text="Mean effective stress, p' (kPa)", range=[0,  1.15 * p_max], row=main_x, col=main_y)

fig.update_yaxes(title_text="q (kPa)", range=[q_range[0], q_range[1] + q_buffer], row=a_x, col=a_y)  
fig.update_yaxes(title_text="Volumetric strain, ε<sub>V</sub> (%)", range=[v_range[0], v_range[1] + v_buffer], row=b_x, col=b_y) 
fig.update_yaxes(title_text="Void ratio, e", range=[ec_range[0] - ec_buffer, ec_range[1] + ec_buffer], row=c_x, col=c_y)  

fig.update_xaxes(title_text="Axial strain, ε<sub>1</sub> (%)", title_standoff=0, range=a_range, row=a_x, col=a_y)
fig.update_xaxes(title_text="Axial strain, ε<sub>1</sub> (%)", title_standoff=0, range=a_range, row=b_x, col=b_y)
fig.update_xaxes(title_text="Mean effective stress, p'(kPa, log scale)", type="log", range=lnp_range, title_standoff=0, row=c_x, col=c_y)

fig.update_layout(
    width=900,
    height=600,
    template = "plotly_white",
    font=dict(size=10),
    margin=dict(l=30, r=20, t=30, b=30),
    legend=dict(
        font=dict(size=9),
        x=1.02, y=1, 
        xanchor="left", yanchor="top",
        bgcolor="rgba(255,255,255,0.8)", 
        bordercolor="#ddd", 
        borderwidth=1)
)
fig.update_xaxes(
    tickfont=dict(size=9),
)
fig.update_yaxes(
    tickfont=dict(size=9),
)
fig.update_traces(marker=dict(size=7), line=dict(width=2))
fig.show(renderer="browser")
fig.write_html("Drained_Loose_Log.html")

In [46]:
#Undrained loose graphs
import json
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import numpy as np

with open("data/Undrained_loose_history.json", "r") as file:
    raw_data = json.load(file)

# outputs shortened list with stepnum items
# def data_choose(data, stepnum):  
#     filtered = []
#     for i in np.linspace(0, len(data)-1, stepnum):
#         filtered.append(data[int(np.round(i))])
#     return filtered

#outputs shortened list with stepnum items with log scale
def data_choose(data, stepnum):  
    filtered = []
    base = np.power(len(data), (1/(stepnum-1)))
    for i in range(stepnum):
        filtered.append(data[int(np.round(np.power(base, i)))-1])
    return filtered

# calculates p and q coordinates for yield curve
def yield_curve(pi, M, n=200):
    p = np.linspace(0, pi * np.e, n)
    q = np.zeros_like(p)
    q[1:] = M * p[1:] * (1 + np.log(pi / p[1:]))
    return p, q
    
data= {}                        # slices data set
for col in raw_data:            
    data.update({col: raw_data[col][:]})    
M = 1.45                        # CSL slope
slide_steps = 18                # number of slider steps
initial_step = 0                # step that slider starts on
static_num = 5                  # number of static traces
step_to_indices = {}            # stores and indexes dynamic traces
p_path = []                     # loading path q values
q_path = []                     # loading path q values
p_max = 0                       # maximum x axis p value
q_max = max(data["p_kPa"]) * M  # maximum y axis q value
a, b, c = 1.01, 0.087, 0.38     # parameters for curved CSL in void ratio space
pref = 100                      # reference pressure for CSL                       
pressure = []                   # pore water pressure for isotropic conditions
for step in data["step"]:
    pressure.append(data["p_kPa"][0] + (data["q_kPa"][step-1]) / 3 - data["p_kPa"][step-1])
main_x, main_y = 1, 1           # layout positions of graphs (row,col)
a_x, a_y = 1, 2
b_x, b_y = 2, 2
c_x, c_y = 3, 2

# Initialize figure with subplots
fig = make_subplots(
    rows=3, cols=2,
    column_widths=[0.7, 0.3],
    horizontal_spacing=0.08,   # space between columns
    vertical_spacing=0.1,     # space between rows
    specs=[[{"rowspan": 3}, {}],
           [     None,      {}],
           [     None,      {}]],
)

# Static traces - always visible
fig.add_trace(go.Scatter(x=[0, 2.5 * q_max / M], y=[0, 2.5 * q_max], visible=True, name="CSL", mode = "lines", line = dict(color="blue", dash='dot')), row=main_x, col=main_y)
fig.add_trace(go.Scatter(x=data["axial_strain_percent"], y=data["q_kPa"], visible=True, name="Loading Path", showlegend=False, line=dict(color="#282828")), row=a_x, col=a_y)
fig.add_trace(go.Scatter(x=data["axial_strain_percent"], y=pressure, visible=True, name="Loading Path", showlegend=False, line=dict(color="#282828")), row=b_x, col=b_y)
ec = []
for p in data["p_kPa"]:
    ec.append(a - b * pow((p / pref), c))
fig.add_trace(go.Scatter(x=data["p_kPa"], y=ec, visible=True, name="CSL", showlegend=False, mode = "lines", line = dict(color = "blue", dash='dot')), row=c_x, col=c_y)
fig.add_trace(go.Scatter(x=data["p_kPa"], y=data["void_ratio"], visible=True, name="Loading Path", showlegend=False, line=dict(color="#282828")), row=c_x, col=c_y)

# Slider-controlled traces
for k, step in enumerate(data_choose(data["step"], slide_steps)):
    p_yield, q_yield = yield_curve(data["pi_kPa"][step-1], M)
    if max(p_yield) > p_max:
        p_max = max(p_yield)
    idx_start = len(fig.data)
    fig.add_trace(go.Scatter(
        visible = False,
        name = "Yield Surface",
        line=dict(color="purple"),
        x = p_yield,
        y = q_yield
    ),
        row=main_x, col=main_y
    )
    
    current_p = data["p_kPa"][step-1]
    current_q = data["q_kPa"][step-1]
    for i in range(step):
        p_path = data["p_kPa"][0:i+1]
        q_path = data["q_kPa"][0:i+1]
    fig.add_trace(go.Scatter(
        visible = False,
        name = "Current State",
        mode="markers",
        marker=dict(size=10, color="red"),
        x = [current_p],
        y = [current_q]
    ),
    row=main_x, col=main_y      
    )
    
    fig.add_trace(go.Scatter(
        visible = False,
        name = "Loading Path",
        mode = 'lines',
        line=dict(color="#282828"),
        x = p_path,
        y = q_path
    ),
    row=main_x, col=main_y      
    )
    
    fig.add_trace(go.Scatter(
        visible = False,
        name = "Image Point",
        mode="markers",
        marker=dict(size=10, color="green"),
        x = [data["pi_kPa"][step-1]],
        y = [data["pi_kPa"][step-1] * M]
    ),
    row=main_x, col=main_y             
    )
    
    fig.add_trace(go.Scatter(
        visible = False,
        name = "Current State",
        mode="markers",
        marker=dict(
            size=12,
            symbol="circle-open",
            color="red",       
            line=dict(width=2)
        ),
        showlegend=False,
        x = [data["axial_strain_percent"][step-1]],
        y = [data["q_kPa"][step-1]]
    ),
    row=a_x, col=a_y            
    )

    fig.add_trace(go.Scatter(
        visible = False,
        name = "Current State",
        mode="markers",
        marker=dict(
            size=12,
            symbol="circle-open",
            color="red",       
            line=dict(width=2)
        ),
        showlegend=False,
        x = [data["axial_strain_percent"][step-1]],
        y = [pressure[step-1]]
    ),
    row=b_x, col=b_y            
    )
    
    fig.add_trace(go.Scatter(
        visible = False,
        name = "Current State",
        mode="markers",
        marker=dict(
            size=12,
            symbol="circle-open",
            color="red",       
            line=dict(width=2)
        ),
        showlegend=False,
        x = [data["p_kPa"][step-1]],
        y = [data["void_ratio"][step-1]]
    ),
    row=c_x, col=c_y             
    )
    idx_end = len(fig.data)
    step_to_indices[k] = list(range(idx_start, idx_end))

# Initial visible traces
for i in step_to_indices[initial_step]: 
    fig.data[i].visible = True 

# updates visible traces based on slider
steps = []
for k in range(len(step_to_indices)):
    visible = [True] * static_num + [False] * (len(fig.data) - static_num)
    for i in step_to_indices[k]:
        visible[i] = True
    steps.append(dict(method="update", args=[{"visible": visible}], label=f"{fig.data[step_to_indices[k][5]].x[0]:.3f}%"))

sliders = [dict(
    active=initial_step, 
    steps=steps, 
    currentvalue={"prefix": "Axial Strain: "}
    )]
fig.update_layout(
    uirevision="constant",
    sliders=sliders
)

#formatting and calculating axis ranges
q_range = [min(data["q_kPa"]), max(data["q_kPa"])]
q_buffer = 0.15 * (max(data["q_kPa"]) - min(data["q_kPa"]))
ec_range = [min(min(ec), min(data["void_ratio"])), max(max(ec), max(data["void_ratio"]))]
ec_buffer = 0.15 * (max(max(ec), max(data["void_ratio"])) - min(min(ec), min(data["void_ratio"])))
pr_range = [min(pressure), max(pressure)]
pr_buffer = 0.15 * (max(pressure) - min(pressure))
a_range = [min(data["axial_strain_percent"]), max(data["axial_strain_percent"])]
lnp_range = np.log10([min(data["p_kPa"]), max(data["p_kPa"])])
#a_buffer = 0.15 * (max(data["axial_strain_percent"]) - min(data["axial_strain_percent"]))

fig.update_yaxes(title_text="Deviatoric stress, q (kPa)", range=[0,  2.5 * q_max], row=main_x, col=main_y)
fig.update_xaxes(title_text="Mean effective stress, p' (kPa)", range=[0,  1.15 * p_max], row=main_x, col=main_y)

fig.update_yaxes(title_text="q (kPa)", range=[q_range[0], q_range[1] + q_buffer], row=a_x, col=a_y) 
fig.update_yaxes(title_text="Pore-pressure, Δu (kPa)", range=[pr_range[0], pr_range[1] + pr_buffer], row=b_x, col=b_y) 
fig.update_yaxes(title_text="Void ratio, <i>e</i>>", range=[ec_range[0] - ec_buffer, ec_range[1] + ec_buffer], row=c_x, col=c_y)  

fig.update_xaxes(title_text="Axial strain, ε<sub>1</sub> (%)", title_standoff=0, range=a_range, row=a_x, col=a_y)
fig.update_xaxes(title_text="Axial strain, ε<sub>1</sub> (%)", title_standoff=0, range=a_range, row=b_x, col=b_y)
fig.update_xaxes(title_text="Mean effective stress, p'(kPa, log scale)", type="log", range=lnp_range, title_standoff=0, row=c_x, col=c_y)

fig.update_layout(
    width=900,
    height=600,
    template = "plotly_white",
    font=dict(size=10),
    margin=dict(l=30, r=20, t=30, b=30),
    legend=dict(
        font=dict(size=9),
        x=1.02, y=1, 
        xanchor="left", yanchor="top",
        bgcolor="rgba(255,255,255,0.8)", 
        bordercolor="#ddd", 
        borderwidth=1
    )
)
fig.update_xaxes(tickfont=dict(size=9))
fig.update_yaxes(tickfont=dict(size=9))
fig.update_traces(marker=dict(size=7), line=dict(width=2))
fig.show(renderer="browser")
fig.write_html("Undrained_Loose_Log.html")

In [47]:
#Undrained dense graphs
import json
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import numpy as np

with open("data/Undrained_dense_history.json", "r") as file:
    raw_data = json.load(file)

# outputs shortened list with stepnum items
# def data_choose(data, stepnum):  
#     filtered = []
#     for i in np.linspace(0, len(data)-1, stepnum):
#         filtered.append(data[int(np.round(i))])
#     return filtered

#outputs shortened list with stepnum items with log scale
def data_choose(data, stepnum):  
    filtered = []
    base = np.power(len(data), (1/(stepnum-1)))
    for i in range(stepnum):
        filtered.append(data[int(np.round(np.power(base, i)))-1])
    return filtered

# calculates p and q coordinates for yield curve
def yield_curve(pi, M, n=200):
    p = np.linspace(0, pi * np.e, n)
    q = np.zeros_like(p)
    q[1:] = M * p[1:] * (1 + np.log(pi / p[1:]))
    return p, q
    
data= {}                        # slices data set
for col in raw_data:            
    data.update({col: raw_data[col][:]})    
M = 1.45                        # CSL slope
slide_steps = 18                # number of slider steps
initial_step = 0                # step that slider starts on
static_num = 5                  # number of static traces
step_to_indices = {}            # stores and indexes dynamic traces
p_path = []                     # loading path q values
q_path = []                     # loading path q values
p_max = 0                       # maximum x axis p value
q_max = max(data["p_kPa"]) * M  # maximum y axis q value
a, b, c = 1.01, 0.087, 0.38     # parameters for curved CSL in void ratio space
pref = 100                      # reference pressure for CSL                       
pressure = []                   # pore water pressure for isotropic conditions
for step in data["step"]:
    pressure.append(data["p_kPa"][0] + (data["q_kPa"][step-1]) / 3 - data["p_kPa"][step-1])
main_x, main_y = 1, 1           # layout positions of graphs (row,col)
a_x, a_y = 1, 2
b_x, b_y = 2, 2
c_x, c_y = 3, 2

# Initialize figure with subplots
fig = make_subplots(
    rows=3, cols=2,
    column_widths=[0.7, 0.3],
    horizontal_spacing=0.08,   # space between columns
    vertical_spacing=0.1,     # space between rows
    specs=[[{"rowspan": 3}, {}],
           [     None,      {}],
           [     None,      {}]],
)

# Static traces - always visible
fig.add_trace(go.Scatter(x=[0, 2.5 * q_max / M], y=[0, 2.5 * q_max], visible=True, name="CSL", mode = "lines", line = dict(color="blue", dash='dot')), row=main_x, col=main_y)
fig.add_trace(go.Scatter(x=data["axial_strain_percent"], y=data["q_kPa"], visible=True, name="Loading Path", showlegend=False, line=dict(color="#282828")), row=a_x, col=a_y)
fig.add_trace(go.Scatter(x=data["axial_strain_percent"], y=pressure, visible=True, name="Loading Path", showlegend=False, line=dict(color="#282828")), row=b_x, col=b_y)
ec = []
for p in data["p_kPa"]:
    ec.append(a - b * pow((p / pref), c))
fig.add_trace(go.Scatter(x=data["p_kPa"], y=ec, visible=True, name="CSL", showlegend=False, mode = "lines", line = dict(color = "blue", dash='dot')), row=c_x, col=c_y)
fig.add_trace(go.Scatter(x=data["p_kPa"], y=data["void_ratio"], visible=True, name="Loading Path", showlegend=False, line=dict(color="#282828")), row=c_x, col=c_y)

# Slider-controlled traces
for k, step in enumerate(data_choose(data["step"], slide_steps)):
    p_yield, q_yield = yield_curve(data["pi_kPa"][step-1], M)
    if max(p_yield) > p_max:
        p_max = max(p_yield)
    idx_start = len(fig.data)
    fig.add_trace(go.Scatter(
        visible = False,
        name = "Yield Surface",
        line=dict(color="purple"),
        x = p_yield,
        y = q_yield
    ),
        row=main_x, col=main_y
    )
    
    current_p = data["p_kPa"][step-1]
    current_q = data["q_kPa"][step-1]
    for i in range(step):
        p_path = data["p_kPa"][0:i+1]
        q_path = data["q_kPa"][0:i+1]
    fig.add_trace(go.Scatter(
        visible = False,
        name = "Current State",
        mode="markers",
        marker=dict(size=10, color="red"),
        x = [current_p],
        y = [current_q]
    ),
    row=main_x, col=main_y      
    )
    
    fig.add_trace(go.Scatter(
        visible = False,
        name = "Loading Path",
        mode = 'lines',
        line=dict(color="#282828"),
        x = p_path,
        y = q_path
    ),
    row=main_x, col=main_y      
    )
    
    fig.add_trace(go.Scatter(
        visible = False,
        name = "Image Point",
        mode="markers",
        marker=dict(size=10, color="green"),
        x = [data["pi_kPa"][step-1]],
        y = [data["pi_kPa"][step-1] * M]
    ),
    row=main_x, col=main_y             
    )
    
    fig.add_trace(go.Scatter(
        visible = False,
        name = "Current State",
        mode="markers",
        marker=dict(
            size=12,
            symbol="circle-open",
            color="red",       
            line=dict(width=2)
        ),
        showlegend=False,
        x = [data["axial_strain_percent"][step-1]],
        y = [data["q_kPa"][step-1]]
    ),
    row=a_x, col=a_y            
    )

    fig.add_trace(go.Scatter(
        visible = False,
        name = "Current State",
        mode="markers",
        marker=dict(
            size=12,
            symbol="circle-open",
            color="red",       
            line=dict(width=2)
        ),
        showlegend=False,
        x = [data["axial_strain_percent"][step-1]],
        y = [pressure[step-1]]
    ),
    row=b_x, col=b_y            
    )
    
    fig.add_trace(go.Scatter(
        visible = False,
        name = "Current State",
        mode="markers",
        marker=dict(
            size=12,
            symbol="circle-open",
            color="red",       
            line=dict(width=2)
        ),
        showlegend=False,
        x = [data["p_kPa"][step-1]],
        y = [data["void_ratio"][step-1]]
    ),
    row=c_x, col=c_y             
    )
    idx_end = len(fig.data)
    step_to_indices[k] = list(range(idx_start, idx_end))

# Initial visible traces
for i in step_to_indices[initial_step]: 
    fig.data[i].visible = True 

# updates visible traces based on slider
steps = []
for k in range(len(step_to_indices)):
    visible = [True] * static_num + [False] * (len(fig.data) - static_num)
    for i in step_to_indices[k]:
        visible[i] = True
    steps.append(dict(method="update", args=[{"visible": visible}], label=f"{fig.data[step_to_indices[k][5]].x[0]:.3f}%"))
    
sliders = [dict(
    active=initial_step, 
    steps=steps, 
    currentvalue={"prefix": "Axial Strain: "}
    )]
fig.update_layout(
    uirevision="constant",
    sliders=sliders
)

#formatting and calculating axis ranges
q_range = [min(data["q_kPa"]), max(data["q_kPa"])]
q_buffer = 0.15 * (max(data["q_kPa"]) - min(data["q_kPa"]))
ec_range = [min(min(ec), min(data["void_ratio"])), max(max(ec), max(data["void_ratio"]))]
ec_buffer = 0.15 * (max(max(ec), max(data["void_ratio"])) - min(min(ec), min(data["void_ratio"])))
pr_range = [min(pressure), max(pressure)]
pr_buffer = 0.15 * (max(pressure) - min(pressure))
a_range = [min(data["axial_strain_percent"]), max(data["axial_strain_percent"])]
lnp_range = np.log10([min(data["p_kPa"]), max(data["p_kPa"])])
#a_buffer = 0.15 * (max(data["axial_strain_percent"]) - min(data["axial_strain_percent"]))

fig.update_yaxes(title_text="Deviatoric stress, q (kPa)", range=[0,  2.5 * q_max], row=main_x, col=main_y)
fig.update_xaxes(title_text="Mean effective stress, p' (kPa)", range=[0,  1.15 * p_max], row=main_x, col=main_y)

fig.update_yaxes(title_text="q (kPa)", range=[q_range[0], q_range[1] + q_buffer], row=a_x, col=a_y) 
fig.update_yaxes(title_text="Pore-pressure, Δu (kPa)", range=[pr_range[0], pr_range[1] + pr_buffer], row=b_x, col=b_y) 
fig.update_yaxes(title_text="Void ratio, <i>e</i>", range=[ec_range[0] - ec_buffer, ec_range[1] + ec_buffer], row=c_x, col=c_y)  

fig.update_xaxes(title_text="Axial strain, ε<sub>1</sub> (%)", title_standoff=0, range=a_range, row=a_x, col=a_y)
fig.update_xaxes(title_text="Axial strain, ε<sub>1</sub> (%)", title_standoff=0, range=a_range, row=b_x, col=b_y)
fig.update_xaxes(title_text="Mean effective stress, p'(kPa, log scale)", type="log", range=lnp_range, title_standoff=0, row=c_x, col=c_y)

fig.update_layout(
    width=900,
    height=600,
    template = "plotly_white",
    font=dict(size=10),
    margin=dict(l=30, r=20, t=30, b=30),
    legend=dict(
        font=dict(size=9),
        x=1.02, y=1, 
        xanchor="left", yanchor="top",
        bgcolor="rgba(255,255,255,0.8)", 
        bordercolor="#ddd", 
        borderwidth=1
    )
)
fig.update_xaxes(tickfont=dict(size=9))
fig.update_yaxes(tickfont=dict(size=9))
fig.update_traces(marker=dict(size=7), line=dict(width=2))
fig.show(renderer="browser")
fig.write_html("Undrained_Dense_Log.html")

In [5]:
a = [1,2,3,4,5]
print(a[:])

[1, 2, 3, 4, 5]


In [18]:
print(np.logspace(np.log10(1),np.log10(3700),num=18))

print(data_choose_log(range(

[2.00000000e+00 3.11326843e+00 4.84622016e+00 7.54379212e+00
 1.17429249e+01 1.82794387e+01 2.84543998e+01 4.42930923e+01
 6.89481430e+01 1.07327039e+02 1.67068940e+02 2.60065229e+02
 4.04826434e+02 6.30166678e+02 9.80939013e+02 1.52696323e+03
 2.37692321e+03 3.70000000e+03]


In [18]:
def data_choose_log(data, stepnum):  
    filtered = []
    last_i = []
    base = np.power(len(data), (1/(stepnum-1)))
    for i in range(stepnum):
        next_i = int(np.round(np.power(base, i)))-1
        while next_i in last_i and not(next_i >= len(data)-2):
            next_i += 1
        filtered.append(data[next_i])
        last_i.append(next_i)
    return filtered

print(data_choose_log(range(3700), 100))

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 48, 53, 57, 62, 68, 74, 80, 87, 95, 103, 112, 122, 133, 144, 157, 171, 185, 202, 219, 238, 259, 281, 306, 332, 361, 393, 427, 464, 504, 548, 595, 647, 703, 764, 830, 902, 980, 1065, 1157, 1257, 1366, 1484, 1613, 1752, 1904, 2069, 2248, 2442, 2654, 2884, 3133, 3404, 3699]


In [ ]:
#all graphs
import json
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import numpy as np

with open("data/Drained_dense_history.json", "r") as file:
    raw_data = json.load(file)

#outputs shortened list with stepnum items
def data_choose(data, stepnum):  
    filtered = []
    for i in np.linspace(0, len(data)-1, stepnum):
        filtered.append(data[int(np.round(i))])
    return filtered

#outputs shortened list with stepnum items with log scale
def data_choose_log(data, stepnum):  
    filtered = []
    last_i = []
    base = np.power(len(data), (1/(stepnum-1)))
    for i in range(stepnum):
        next_i = int(np.round(np.power(base, i)))-1
        while next_i in last_i and not(next_i >= len(data)-2):
            next_i += 1
        filtered.append(data[next_i])
        last_i.append(next_i)
    return filtered
    
# calculates p and q coordinates for yield curve
def yield_curve(pi, M, n=50):
    p = np.linspace(0, pi * np.e, n)
    q = np.zeros_like(p)
    q[1:] = M * p[1:] * (1 + np.log(pi / p[1:]))
    return p, q

data={}
short = 50
for col in raw_data:            # shortens data set
    data.update({col: data_choose_log(raw_data[col], short)})
mtc = PARAMS.mtc                # CSL slope
slide_steps = 18                # number of slider steps
initial_step = 0                # step that slider starts on
static_num = 5                  # number of static traces
step_to_indices = {}            # stores and indexes dynamic traces
p_path = []                     # loading path p values
q_path = []                     # loading path q values
p_max = 0                       # maximum x axis p value, currently 0 but changes based on data
q_max = max(data["p_kpa"]) * mtc  # maximum y axis q value
a, b, c = PARAMS.ca, PARAMS.cb, PARAMS.cc     # parameters for curved CSL in void ratio space
lam, gam = PARAMS.lambda10, PARAMS.gamma      # parameters for semi-log CSL in void ratio space
pref = PARAMS.pref              # reference pressure for CSL
pressure = []                   # pore water pressure for isotropic conditions
for step in data["step"]:
    pressure.append(data["p_kpa"][0] + (data["q_kpa"][step]) / 3 - data["p_kpa"][step])
main_x, main_y = 1, 1           # layout positions of graphs (row,col)
a_x, a_y = 1, 2
b_x, b_y = 2, 2
c_x, c_y = 3, 2

# Initialize figure with subplots
fig = make_subplots(
    rows=3, cols=2,
    column_widths=[0.7, 0.3],
    horizontal_spacing=0.08,   # space between columns
    vertical_spacing=0.1,      # space between rows
    specs=[[{"rowspan": 3}, {}],
           [     None,      {}],
           [     None,      {}]],
)

# Static traces - always visible
fig.add_trace(go.Scatter(x=[0, 2.5 * q_max / mtc], y=[0, 2.5 * q_max], visible=True, name="CSL", mode="lines", line=dict(dash='dot', color="blue")), row=main_x, col=main_y)
fig.add_trace(go.Scatter(x=data["eps1_pct"], y=data["q_kpa"], visible=True, name="Loading Path", showlegend=False, line=dict(color="#282828")), row=a_x, col=a_y)
if PARAMS.drainage == "drained":
    fig.add_trace(go.Scatter(x=data["eps1_pct"], y=data["epsv_pct"], visible=True, name="Loading Path", showlegend=False, line=dict(color="#282828")), row=b_x, col=b_y)
elif PARAMS.drainage == "undrained":
    fig.add_trace(go.Scatter(x=data["eps1_pct"], y=pressure, visible=True, name="Loading Path", showlegend=False, line=dict(color="#282828")), row=b_x, col=b_y)
ec = []
if(PARAMS.csl_type == "curved"):
    for p in data["p_kpa"]:
        ec.append(a - b * pow((p / pref), c))
elif(PARAMS.csl_type == "semi_log"):
    for p in data["p_kpa"]:
        ec.append(gam - lam * np.log10(p / pref))
fig.add_trace(go.Scatter(x=data["p_kpa"], y=ec, visible=True, name="CSL", showlegend=False, mode="lines", line=dict(color="blue", dash='dot')), row=c_x, col=c_y)
fig.add_trace(go.Scatter(x=data["p_kpa"], y=data["e"], visible=True, name="Loading Path", showlegend=False, line=dict(color="#282828")), row=c_x, col=c_y)

# Slider-controlled traces
for k, step in enumerate(data_choose_log(data["step"], slide_steps)):
    p_yield, q_yield = yield_curve(data["p_i_kpa"][step], data["Mi"][step])
    if max(p_yield) > p_max:
        p_max = max(p_yield)
    idx_start = len(fig.data)
    fig.add_trace(go.Scatter(
        visible=False,
        name="Yield Surface",
        line=dict(color="purple"),
        x=p_yield,
        y=q_yield
    ),
        row=main_x, col=main_y
    )

    current_p = data["p_kpa"][step]
    current_q = data["q_kpa"][step]
    for i in range(step):
        p_path = data["p_kpa"][0:i + 1]
        q_path = data["q_kpa"][0:i + 1]
    fig.add_trace(go.Scatter(
        visible=False,
        name="Current State",
        mode="markers",
        marker=dict(size=10, color="red"),
        x=[current_p],
        y=[current_q]
    ),
        row=main_x, col=main_y
    )

    fig.add_trace(go.Scatter(
        visible=False,
        name="Loading Path",
        mode='lines',
        line=dict(color="#282828"),
        x=p_path,
        y=q_path
    ),
        row=main_x, col=main_y
    )

    fig.add_trace(go.Scatter(
        visible=False,
        name="Image Point",
        mode="markers",
        marker=dict(size=10, color="green"),
        x=[data["p_i_kpa"][step]],
        y=[data["p_i_kpa"][step] * data["Mi"][step]]
    ),
        row=main_x, col=main_y
    )

    fig.add_trace(go.Scatter(
        visible=False,
        name="Current State",
        mode="markers",
        marker=dict(
            size=12,
            symbol="circle-open",
            color="red",
            line=dict(width=2)
        ),
        showlegend=False,
        x=[data["eps1_pct"][step]],
        y=[data["q_kpa"][step]]
    ),
        row=a_x, col=a_y
    )
    if PARAMS.drainage == "drained":
        fig.add_trace(go.Scatter(
            visible=False,
            name="Current State",
            mode="markers",
            marker=dict(
                size=12,
                symbol="circle-open",
                color="red",
                line=dict(width=2)
            ),
            showlegend=False,
            x=[data["eps1_pct"][step]],
            y=[data["epsv_pct"][step]]
        ),
            row=b_x, col=b_y
        )
    elif PARAMS.drainage == "undrained":
        fig.add_trace(go.Scatter(
            visible = False,
            name = "Current State",
            mode="markers",
            marker=dict(
                size=12,
                symbol="circle-open",
                color="red",       
                line=dict(width=2)
            ),
            showlegend=False,
            x = [data["eps1_pct"][step]],
            y = [pressure[step]]
        ),
        row=b_x, col=b_y            
        )
    fig.add_trace(go.Scatter(
        visible=False,
        name="Current State",
        mode="markers",
        marker=dict(
            size=12,
            symbol="circle-open",
            color="red",
            line=dict(width=2)
        ),
        showlegend=False,
        x=[data["p_kpa"][step]],
        y=[data["e"][step]]
    ),
        row=c_x, col=c_y
    )
    idx_end = len(fig.data)
    step_to_indices[k] = list(range(idx_start, idx_end))

# Initial visible traces
for i in step_to_indices[initial_step]:
    fig.data[i].visible = True

# updates visible traces based on slider
steps = []
for k in range(len(step_to_indices)):
    visible = [True] * static_num + [False] * (len(fig.data) - static_num)
    for i in step_to_indices[k]:
        visible[i] = True
    steps.append(dict(method="update", args=[{"visible": visible}], label=f"{fig.data[step_to_indices[k][5]].x[0]:.3f}%"))

sliders = [dict(
    active=initial_step,
    steps=steps,
    currentvalue={"prefix": "Axial Strain: "}
)]
fig.update_layout(
    uirevision="constant",
    sliders=sliders
)

# formatting and calculating axis ranges
q_range = [min(data["q_kpa"]), max(data["q_kpa"])]
q_buffer = 0.15 * (max(data["q_kpa"]) - min(data["q_kpa"]))
ec_range = [min(min(ec), min(data["e"])), max(max(ec), max(data["e"]))]
ec_buffer = 0.15 * (max(max(ec), max(data["e"])) - min(min(ec), min(data["e"])))
v_range = [min(data["epsv_pct"]), max(data["epsv_pct"])]
v_buffer = 0.15 * (max(data["epsv_pct"]) - min(data["epsv_pct"]))
a_range = [min(data["eps1_pct"]), max(data["eps1_pct"])]
lnp_range = np.log10([min(data["p_kpa"]), max(data["p_kpa"])]) #compensates for plotly only accepting log axis ranges in log10
pr_range = [min(pressure), max(pressure)]
pr_buffer = 0.15 * (max(pressure) - min(pressure))

fig.update_yaxes(title_text="Deviatoric stress, q (kPa)", range=[0, 2.5 * q_max], row=main_x, col=main_y)
fig.update_xaxes(title_text="Mean effective stress, p' (kPa)", range=[0, 1.15 * p_max], row=main_x, col=main_y)

fig.update_yaxes(title_text="q (kPa)", range=[q_range[0], q_range[1] + q_buffer], row=a_x, col=a_y)
fig.update_xaxes(title_text="Axial strain, ε<sub>1</sub> (%)", title_standoff=0, range=a_range, row=a_x, col=a_y)

if PARAMS.drainage == "drained":
    fig.update_yaxes(title_text="Volumetric strain, ε<sub>V</sub> (%)", range=[v_range[0], v_range[1] + v_buffer], row=b_x, col=b_y)
    fig.update_xaxes(title_text="Axial strain, ε<sub>1</sub> (%)", title_standoff=0, range=a_range, row=b_x, col=b_y)
elif PARAMS.drainage == "undrained":
    fig.update_yaxes(title_text="Pore-pressure, Δu (kPa)", range=[pr_range[0], pr_range[1] + pr_buffer], row=b_x, col=b_y)
    fig.update_xaxes(title_text="Axial strain, ε<sub>1</sub> (%)", title_standoff=0, range=a_range, row=b_x, col=b_y)

fig.update_yaxes(title_text="Void ratio, e", range=[ec_range[0] - ec_buffer, ec_range[1] + ec_buffer], row=c_x, col=c_y)
fig.update_xaxes(title_text="Mean effective stress, p'(kPa, log scale)", type="log", range=lnp_range, title_standoff=0, row=c_x, col=c_y)

fig.update_layout(
    width=900,
    height=600,
    template="plotly_white",
    font=dict(size=10),
    margin=dict(l=30, r=20, t=30, b=30),
    legend=dict(
        font=dict(size=9),
        x=1.02, y=1,
        xanchor="left", yanchor="top",
        bgcolor="rgba(255,255,255,0.8)",
        bordercolor="#ddd",
        borderwidth=1)
)
fig.update_xaxes(
    tickfont=dict(size=9),
)
fig.update_yaxes(
    tickfont=dict(size=9),
)
fig.update_traces(marker=dict(size=7), line=dict(width=2))
fig.show(renderer="browser")